# Zero-Shot LLM Stance Classification -- Jupyter version (LM Studio / Ollama)

Run these cells **top to bottom**, in order. No terminal needed.

**Before you start:**
1. Put `stance_processed.csv` in the **same folder** as this notebook.
2. Open the LM Studio app, go to the **Developer** tab, click **Start Server**.
   Leave that app open in the background.
3. Open `http://localhost:1234/v1/models` in a browser tab to see the exact
   model id strings you'll need in Cell 3 below.

If you're using **Ollama** instead of LM Studio, skip the "Start Server" step
(Ollama runs automatically after install) and use the simpler model tags
already filled in below (`llama3.1:8b`, etc.) instead of LM Studio's ids.


## Cell 1 -- Install the packages this notebook needs (one time only)

In [ ]:
import sys
!{sys.executable} -m pip install -q openai pandas scikit-learn matplotlib
print("Done installing.")


## Cell 2 -- Settings you can edit
This is the only cell you normally need to change. Fill in your model ids
from `http://localhost:1234/v1/models` (LM Studio), or leave the Ollama
tags as-is if you're using Ollama.

In [ ]:
# ---- EDIT THESE ----
CSV_PATH = "stance_processed.csv"   # must be in the same folder as this notebook
TEXT_COL = "text"
LABEL_COL = "label"

# Pick ONE of the two lines below depending on which tool you're using:
BASE_URL = "http://localhost:1234/v1"   # LM Studio
# BASE_URL = "http://localhost:11434/v1"  # Ollama (uncomment this line instead if using Ollama)

# LM Studio: paste the exact ids from http://localhost:1234/v1/models
# Ollama: these tags already work as-is (no need to change)
MODELS = {
    "Llama-3.1-8B": "llama3.1:8b",
    "Llama-3.2-3B": "llama3.2:3b",
    "Qwen2.5-7B":   "qwen2.5:7b",
    "Gemma-2-9B":   "gemma2:9b",
    "Mistral-7B":   "mistral:7b",
}

TEST_SIZE = 0.2   # 20% held out for testing
SEED = 42
LIMIT = 20        # set to None for the full test set once the quick test below works
# ---------------------
print("Settings loaded. LIMIT =", LIMIT, "(remember to set this to None for the full run)")


## Cell 3 -- Helper functions (just run this, no need to edit)

In [ ]:
import re, time, json, unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from openai import OpenAI

LABELS = ["pro_public", "pro_private", "neutral"]

PROMPT_TEMPLATE = (
    "You are given a Bengali social media comment about the debate between "
    "public and private universities in Bangladesh.\n"
    "Classify its stance as exactly one label: pro_public, pro_private, or neutral.\n"
    "pro_public = favors public universities. pro_private = favors private "
    "universities. neutral = no clear preference or purely informational.\n"
    "Comment: \"{text}\"\n"
    "Answer with exactly one word and nothing else: pro_public, pro_private, or neutral."
)

EMOJI_PATTERN = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF\U00002700-\U000027BF\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF]+", flags=re.UNICODE)
ENGLISH_PATTERN = re.compile(r"[a-zA-Z]+")
PUNCT_PATTERN = re.compile(r"[^\u0980-\u09FF\s]")
MULTISPACE = re.compile(r"\s+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = ENGLISH_PATTERN.sub(" ", text)
    text = PUNCT_PATTERN.sub(" ", text)
    return MULTISPACE.sub(" ", text).strip()

def parse_label(generated_text: str) -> str:
    t = generated_text.strip().lower()
    for lbl in LABELS:
        if lbl in t or lbl.replace("_", " ") in t or lbl.replace("_", "-") in t:
            return lbl
    has_public, has_private = "public" in t, "private" in t
    if has_public and not has_private:
        return "pro_public"
    if has_private and not has_public:
        return "pro_private"
    return "neutral"

def classify_one(client, model_id, text, max_retries=3):
    prompt = PROMPT_TEMPLATE.format(text=text)
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=8,
                temperature=0.0,
            )
            return parse_label(resp.choices[0].message.content)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  [warning] gave up on one row after {max_retries} tries: {e}")
                return "neutral"
            time.sleep(2)

print("Helper functions ready.")


## Cell 4 -- Load and clean your CSV, carve out the test set
This reads your CSV, does light cleaning (removes emoji/English letters/
punctuation, drops empty/duplicate rows), and holds out a stratified 20%
test split -- the same rows every time (fixed seed), so results are
reproducible.

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw["clean_text"] = raw[TEXT_COL].apply(clean_text)
raw = raw[raw["clean_text"].str.len() > 0].drop_duplicates(subset=["clean_text"])

_, test_df = train_test_split(raw, test_size=TEST_SIZE, random_state=SEED, stratify=raw[LABEL_COL])
test_df = test_df.reset_index(drop=True)

if LIMIT:
    test_df = test_df.head(LIMIT)

print(f"Test set: {len(test_df)} rows")
test_df[LABEL_COL].value_counts()


## Cell 5 -- Connect to your local model server
This just opens a connection -- doesn't call any model yet. If this cell
errors, LM Studio's server (or Ollama) isn't running, or BASE_URL is wrong.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key="not-needed")
print(f"Connected to {BASE_URL}")


## Cell 6 -- Quick sanity check on ONE row, ONE model
Run this before the full loop below, so you catch a wrong model id or a
server problem on a single call instead of waiting through a whole run.

In [ ]:
sample_text = test_df["clean_text"].iloc[0]
sample_model_key = list(MODELS.keys())[0]
sample_model_id = MODELS[sample_model_key]

print("Text:", sample_text)
print("Model:", sample_model_key, "->", sample_model_id)
result = classify_one(client, sample_model_id, sample_text)
print("Predicted label:", result)


## Cell 7 -- Run all models on the test set
This is the slow part. Progress prints every 50 rows per model. With
`LIMIT = 20` (from Cell 2) this should be quick; once you're confident it
works, go back to Cell 2, set `LIMIT = None`, and re-run Cells 4-7 for the
full test set.

In [ ]:
results = test_df[["clean_text", LABEL_COL]].copy()
summary_rows = []

for name, model_id in MODELS.items():
    print(f"\n=== Running {name} ({model_id}) ===")
    preds = []
    t0 = time.time()
    for i, text in enumerate(test_df["clean_text"]):
        preds.append(classify_one(client, model_id, text))
        if i % 50 == 0:
            print(f"  {i}/{len(test_df)}  ({time.time()-t0:.0f}s elapsed)")

    results[f"pred_{name}"] = preds
    y_true = test_df[LABEL_COL].values
    acc = accuracy_score(y_true, preds)
    mf1 = f1_score(y_true, preds, average="macro")
    summary_rows.append({"model": name, "accuracy": acc, "macro_f1": mf1})
    print(f"{name}: accuracy={acc:.4f} macro_f1={mf1:.4f} ({time.time()-t0:.0f}s total)")
    print(classification_report(y_true, preds))

import os
os.makedirs("results", exist_ok=True)
results.to_csv("results/llm_zeroshot_predictions_local.csv", index=False)
pd.DataFrame(summary_rows).to_csv("results/llm_zeroshot_summary_local.csv", index=False)
print("\nSaved to results/llm_zeroshot_predictions_local.csv")


## Cell 8 -- Build the formatted results table (same layout as your screenshot)
Shows the table inline AND saves it as a PNG image + CSV.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

LABEL_ORDER = ["pro_public", "pro_private", "neutral"]
LABEL_DISPLAY = {"pro_public": "Pro-Public", "pro_private": "Pro-Private", "neutral": "Neutral"}

pred_cols = {c.replace("pred_", ""): c for c in results.columns if c.startswith("pred_")}
y_true_all = results[LABEL_COL].values

rows = []
for model_name, col in pred_cols.items():
    y_pred = results[col].values
    p, r, f1, support = precision_recall_fscore_support(y_true_all, y_pred, labels=LABEL_ORDER, zero_division=0)
    acc = accuracy_score(y_true_all, y_pred) * 100
    macro_f1 = f1_score(y_true_all, y_pred, average="macro", labels=LABEL_ORDER) * 100
    for i, lbl in enumerate(LABEL_ORDER):
        rows.append({
            "Type": "LLM", "Model": model_name, "Class": LABEL_DISPLAY[lbl],
            "Precision(%)": round(p[i]*100, 2), "Recall(%)": round(r[i]*100, 2),
            "F1(%)": round(f1[i]*100, 2), "Overall Acc.(%)": round(acc, 2),
            "Macro F1(%)": round(macro_f1, 2),
        })

table_df = pd.DataFrame(rows)
table_df.to_csv("results/llm_results_table.csv", index=False)

# paper-style view: blank Overall Acc./Macro F1 on rows 2-3 of each model block
view = table_df.copy()
for col in ["Overall Acc.(%)", "Macro F1(%)"]:
    view[col] = view[col].astype(object)
    view.loc[view["Model"] == view["Model"].shift(), col] = ""
view


## Cell 9 -- Render it as an image (matches your screenshot's look)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

col_labels = list(view.columns)
cell_text = view.astype(str).values.tolist()
fig_h = 0.42 * len(view) + 0.6
fig, ax = plt.subplots(figsize=(11, fig_h))
ax.axis("off")
ax.set_title("Zero-Shot LLM Stance Classification Results", fontsize=12, fontweight="bold", pad=12)
table = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
for j in range(len(col_labels)):
    table[0, j].set_text_props(fontweight="bold")
    table[0, j].set_facecolor("#e8e8e8")

numeric_macro = pd.to_numeric(table_df["Macro F1(%)"], errors="coerce")
if numeric_macro.notna().any():
    best_row = numeric_macro.idxmax()
    table[best_row + 1, col_labels.index("Macro F1(%)")].set_text_props(fontweight="bold")

plt.tight_layout()
plt.savefig("results/llm_results_table.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to results/llm_results_table.png")
